# Assignment 11: Image Recognition with CNNs and Transfer Learning

Fill in every `# TODO` and answer every question in the **Your answer** cells. Run all cells before submitting.

This notebook needs PyTorch and torchvision. The easiest way is **Google Colab** (both are already installed). On your laptop: `pip install torch torchvision`. Part A takes about 2–4 minutes on a laptop CPU.

## Part A. Convolutional networks on Fashion-MNIST

Fashion-MNIST has 70,000 grayscale 28×28 images of 10 kinds of clothing. We compare a regular fully connected network (MLP) with a **convolutional neural network (CNN)**.

To keep training fast on a laptop CPU, we use 20,000 training images. The first run downloads about 30 MB.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

torch.manual_seed(0)
to_tensor = transforms.ToTensor()
train_full = datasets.FashionMNIST("data", train=True, download=True, transform=to_tensor)
test_set = datasets.FashionMNIST("data", train=False, download=True, transform=to_tensor)
train_set = Subset(train_full, range(20000))
classes = train_full.classes

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=1000)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, i in zip(axes, range(8)):
    img, label = train_full[i]
    ax.imshow(img[0], cmap="gray_r"); ax.set_title(classes[label], fontsize=9); ax.axis("off")
plt.show()

The helper functions below train a model and measure its accuracy. They work like the training loop you wrote in Assignment 10, with the Adam optimizer.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in loader:
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += len(yb)
    model.train()
    return correct / total

def fit(model, epochs=3, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    start = time.time()
    for epoch in range(epochs):
        for xb, yb in train_loader:
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        print(f"epoch {epoch + 1}: test accuracy {accuracy(model, test_loader):.3f}  ({time.time() - start:.0f}s)")

def n_params(model):
    return sum(p.numel() for p in model.parameters())

### Question 1. A fully connected baseline

Here is an MLP that flattens each image into 784 numbers. Train it for 3 epochs and note its test accuracy and number of parameters.

In [ ]:
torch.manual_seed(0)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
print("MLP parameters:", n_params(mlp))
fit(mlp)

### Question 2. Build a CNN

Complete the CNN below:

- Conv layer: 1 input channel → 16 filters of size 3×3 with `padding=1`, ReLU, then 2×2 max pooling (28×28 → 14×14).
- Conv layer: 16 → 32 filters of size 3×3 with `padding=1`, ReLU, then 2×2 max pooling (14×14 → 7×7).
- Flatten, a fully connected layer with 64 neurons and ReLU, and an output layer with 10 neurons.

Train it for 3 epochs.

In [ ]:
torch.manual_seed(0)
cnn = nn.Sequential(
    # TODO: the layers described above
)
print("CNN parameters:", n_params(cnn))
fit(cnn)

### Question 3. Compare

a. Which model is more accurate? Which has more parameters?

b. Count by hand the parameters of the first convolutional layer (16 filters of size 3×3 on 1 channel, plus one bias per filter). Compare it with the first layer of the MLP (784 inputs → 128 neurons). Why does a convolutional layer need so few parameters?

c. In one or two sentences, explain why a CNN suits images better than an MLP. *Hint:* what does the MLP lose when it flattens the image?

**Your answer:** *Write your answer here.*

### Question 4. What did the first layer learn?

The cell below shows the 16 learned 3×3 filters of the first layer, and what each filter produces for one image (its **feature map**). Describe two filters: what kind of pattern does each one respond to?

In [ ]:
filters = cnn[0].weight.detach()[:, 0]
img, label = test_set[0]
with torch.no_grad():
    maps = torch.relu(cnn[0](img.unsqueeze(0)))[0]

fig, axes = plt.subplots(2, 16, figsize=(16, 2.4))
for k in range(16):
    axes[0, k].imshow(filters[k], cmap="bwr"); axes[0, k].axis("off")
    axes[1, k].imshow(maps[k], cmap="gray_r"); axes[1, k].axis("off")
axes[0, 0].set_title("filters", loc="left"); axes[1, 0].set_title(f"feature maps of a {classes[label]}", loc="left")
plt.show()

**Your answer:** *Write your answer here.*

### Question 5. Which mistakes does the CNN make?

Compute the confusion matrix of the CNN on the test set. Which two classes does the CNN confuse most often? Does that make sense to you?

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cnn.eval()
with torch.no_grad():
    # TODO: collect the CNN's predictions and the true labels for the whole test set
ConfusionMatrixDisplay.from_predictions(truth, preds, display_labels=classes, xticks_rotation=60, colorbar=False)
plt.gcf().set_size_inches(8, 8); plt.show()

**Your answer:** *Write your answer here.*

## Part B. Transfer learning on your own images

Training a CNN from scratch needs a lot of images. **Transfer learning** reuses a large CNN that has already been trained on 1.2 million photos (ResNet-18, trained on ImageNet). We keep its convolutional layers as a fixed **feature extractor** and train only a new last layer for *our* categories.

### Collect your images

Choose **2 to 4 categories** of images you want to recognize (for example: cats vs dogs, Bryant buildings, kinds of food, your handwriting of a/b/c). Collect **at least 30 images per category** and arrange them in folders, one per category:

```
my_images/
    cats/   img1.jpg, img2.jpg, ...
    dogs/   ...
```

Set `DATA_DIR` below to your folder.

In [ ]:
# TODO: set DATA_DIR to the folder with one subfolder per category

# Image types to read. Images saved from Google are often .jfif or .webp, which torchvision skips by default.
IMG_EXT = (".jpg", ".jpeg", ".png", ".jfif", ".webp", ".bmp")
is_image = lambda path: path.lower().endswith(IMG_EXT)

weights = models.ResNet18_Weights.DEFAULT
preprocess = weights.transforms()          # resize, crop and normalize the way ResNet expects
images = datasets.ImageFolder(DATA_DIR, transform=transforms.Compose([transforms.Lambda(lambda im: im.convert("RGB")), preprocess]), is_valid_file=is_image)
print("Categories:", images.classes)
print("Images per category:", np.bincount(images.targets))

### Question 6. Extract features with a pretrained ResNet

ResNet-18 turns each image into 512 numbers just before its last layer. Replace its last layer (`resnet.fc`) with `nn.Identity()` so that the network outputs those 512 features, then compute the features of all your images.

In [ ]:
resnet = models.resnet18(weights=weights)
# TODO: replace the last layer so that the network outputs its 512 features
resnet.eval()

with torch.no_grad():
    feats, labels = [], []
    for xb, yb in DataLoader(images, batch_size=32):
        feats.append(resnet(xb)); labels.append(yb)
features = torch.cat(feats).numpy()
labels = torch.cat(labels).numpy()
print(features.shape)

### Question 7. Train and evaluate your classifier

a. Split the features into 70% training and 30% testing (stratified, `random_state=1`) and train a `LogisticRegression(max_iter=5000)` on the ResNet features. Report the test accuracy.

b. For comparison, train a `RandomForestClassifier(random_state=1)` on the **raw pixels** of the same images (resized to 64×64), with the same split. This is the approach from Assignment 6.

c. Explain the difference in accuracy.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

idx_train, idx_test = train_test_split(np.arange(len(labels)), test_size=0.3, random_state=1, stratify=labels)

# TODO: (a) logistic regression on the ResNet features
print(f"Transfer learning (ResNet features + logistic regression): {transfer_acc:.3f}")

raw = datasets.ImageFolder(DATA_DIR, transform=transforms.Compose([
    transforms.Lambda(lambda im: im.convert("RGB")), transforms.Resize((64, 64)), transforms.ToTensor()]), is_valid_file=is_image)
pixels = torch.stack([x for x, _ in raw]).flatten(1).numpy()
# TODO: (b) random forest on the raw pixels
print(f"Random forest on raw pixels:                                {pixel_acc:.3f}")

**Your answer:** *Write your answer here.*

### Question 8. Look at the mistakes

Show the test images your transfer-learning model got wrong (if any), with their true and predicted labels. Why might the model have struggled with them? If it got everything right, show the 4 test images it was **least confident** about instead.

In [ ]:
proba = clf.predict_proba(features[idx_test])
pred = proba.argmax(1)
confidence = proba.max(1)
wrong = np.where(pred != labels[idx_test])[0]
show = wrong if len(wrong) > 0 else np.argsort(confidence)[:4]
print("Number of mistakes:", len(wrong))

fig, axes = plt.subplots(1, max(len(show), 1), figsize=(3 * max(len(show), 1), 3), squeeze=False)
for ax, k in zip(axes[0], show):
    img, true = raw[idx_test[k]]
    ax.imshow(img.permute(1, 2, 0)); ax.axis("off")
    ax.set_title(f"true {images.classes[true]}\npred {images.classes[pred[k]]} ({confidence[k]:.0%})", fontsize=9)
plt.show()

**Your answer:** *Write your answer here.*